# Orchestrating a Team of Agents

### One safety desk, eight vehicles, 3,976 owner complaints — and the question of how many agents it takes

A single agent with good tools is a strong baseline. It is also, very often, the right
answer. This notebook is about the cases where it is not, and about the thing that replaces
it — not "more agents", but an **architecture**: who decides, who sees what, who checks whom,
and when the whole thing stops.

Every part ends with numbers. By the end there is a table of them, and the last part is
about reading that table and deciding, for your own problem, how many agents you actually need.

## The desk

**Halyard Analytics** monitors vehicle defect reports for insurers and fleet operators. Its
field-safety desk is called **Canary**, and every Monday it has one job:

> For each vehicle on the watchlist, find the three components whose owner complaints look most
> dangerous, say in one sentence what is failing in each, back it with the complaint count and
> report numbers, and make a call — **escalate**, **monitor**, or **close**.

The brief goes to a safety engineer who signs it. If it is wrong, an insurer prices risk on a
defect that isn't there, or misses one that is.

In [1]:
#%pip install -q langchain langgraph langchain-openai langgraph-checkpoint-sqlite openai-agents pandas ipython-autotime

In [2]:
%load_ext autotime

time: 87 µs (started: 2026-09-26 20:49:59 +05:30)


In [3]:
import json
import operator
import sqlite3
import time
import warnings
from typing import Annotated, Literal, TypedDict

import pandas as pd
from pydantic import BaseModel, Field

# Two models, on purpose. Eight analysts read narratives in parallel — that is extraction
# work, and the small model does it. Routing, writing and verifying are judgement, and get
# the larger one. Spending unevenly is one of the things having a team buys you.
WORKER = "gpt-5-nano"
JUDGE = "gpt-5-mini"

# Everything in this notebook runs at the same reasoning effort, so that no comparison
# between two designs is really a comparison between two settings.
EFFORT = "low"

# Structured output makes pydantic grumble about a `parsed` field on every single call.
# The warning says nothing about this notebook's correctness.
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic.main")

time: 306 ms (started: 2026-09-26 20:49:59 +05:30)


### The field reports

**Four tables, built from NHTSA's public complaint and recall feeds.**

| table | rows | what it holds |
|---|---|---|
| `watchlist` | 8 | The vehicles the desk covers, with their complaint and recall counts |
| `complaints` | 3,976 | Owner-filed reports: the narrative, plus crash / fire / injury / death flags |
| `complaint_components` | 5,710 | One row per (complaint, component) — the tallies come from here |
| `recalls` | 99 | Campaigns actually opened, with the component each one covers |

- **The narratives are why a language model is in this system at all,** and why it cannot all
  fit in one place: 2.36 million characters, about 591,000 tokens.
- **Every read in this notebook goes through `sql()`, and `sql()` opens the file read-only.** No
  agent can change the data it is scored against.

In [4]:
DB_PATH = "field_reports.db"


def sql(query, params=(), limit=50):
    """Run one query against the field reports. Returns up to `limit` rows, each a dict.

    `mode=ro` opens the file read-only. A plain sqlite3.connect(path) would happily run a
    DROP TABLE for an agent, so "read-only" has to live in the connection, not in a docstring.
    """
    con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)
    con.row_factory = sqlite3.Row  # rows that know their column names
    rows = [dict(row) for row in con.execute(query, params).fetchmany(limit)]
    con.close()
    return rows

time: 379 µs (started: 2026-09-26 20:49:59 +05:30)


In [5]:
# The watchlist: all eight vehicles the desk covers.
WATCHLIST = [w["vehicle_id"]
             for w in sql("SELECT vehicle_id FROM watchlist ORDER BY complaint_count DESC")]

total = sql("SELECT COUNT(*) n, SUM(LENGTH(narrative)) chars FROM complaints")[0]
print(f"{total['n']} complaints, {total['chars'] / 1e6:.2f}M characters, "
      f"~{total['chars'] / 4000:.0f}k tokens")

pd.DataFrame(sql("SELECT * FROM watchlist ORDER BY complaint_count DESC"))

3976 complaints, 2.36M characters, ~591k tokens


,vehicle_id,make,model,model_year,complaint_count,recall_count
0,ford-f-150-2021,ford,f-150,2021,1006,29
1,honda-accord-2019,honda,accord,2019,685,6
2,tesla-model-3-2021,tesla,model 3,2021,660,22
3,toyota-rav4-2020,toyota,rav4,2020,634,6
4,jeep-grand-cherokee-2021,jeep,grand cherokee,2021,403,12
5,nissan-rogue-2021,nissan,rogue,2021,286,10
6,chevrolet-bolt-ev-2020,chevrolet,bolt ev,2020,172,8
7,hyundai-elantra-2021,hyundai,elantra,2021,130,6


time: 9.09 ms (started: 2026-09-26 20:49:59 +05:30)


In [7]:
pd.options.display.max_colwidth = None

time: 188 µs (started: 2026-09-26 20:50:08 +05:30)


In [8]:
# complaints: the five newest on one vehicle, narrative cut to 70 characters for display.
pd.DataFrame(sql("""
    SELECT odi_number, vehicle_id, date_filed, components, crash, fire, injuries, deaths,
           was_masked, narrative 
      FROM complaints
     WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
     ORDER BY odi_number DESC""", limit=5))

,odi_number,vehicle_id,date_filed,components,crash,fire,injuries,deaths,was_masked,narrative
0,11755357,chevrolet-bolt-ev-2020,08/06/2026,STEERING,0,0,0,0,0,"Steering rack is starting to fail at under 47000 miles, car pulls to right during acceleration and pulls left when decelerating. Steering wheel does not return to the center and a slight left pull on the wheel has to be maintained at all times so the car doesn't merge right into traffic. It is available upon inspection request. I almost didn't realize it pulling to the right and almost merged into another car on the highway. It has not been confirmed yet by a third party. It has not been inspected yet. No warning lights or messages occured when it started happening."
1,11748422,chevrolet-bolt-ev-2020,07/06/2026,FUEL/PROPULSION SYSTEM,0,0,0,0,1,GM installed Advanced Monitoring Software on the car rather than replace the battery. The software detected a battery fault. Under the battery [REDACTED] if a fault is detected the battery is replaced free of charge. GM is refusing to replace the battery. Now the car will not move.
2,11745256,chevrolet-bolt-ev-2020,06/19/2026,"ELECTRICAL SYSTEM,FUEL/PROPULSION SYSTEM",0,0,0,0,1,"General Motors is using the arbitrary 6,213-mile tracking window of [REDACTED] 944 to deny coverage for a diagnosed Cell Section 3 hardware failure at 123,582 miles. However, this vehicle is directly impacted by the failure parameters outlined in the expanded GM Safety [REDACTED] N242470160 / N242470162, where GM openly acknowledges to federal regulators that the advanced diagnostic software has failed to properly detect defective battery modules. By denying a physical battery replacement when a manufacturing defect is caught during a mandatory corporate software reconfiguration, GM is violating the spirit and legal intent of their ongoing federal safety [REDACTED] campaigns."
3,11742825,chevrolet-bolt-ev-2020,06/08/2026,STEERING,0,0,0,0,0,Steering stiffened over several weeks. Then steering wheel would not automatically return to center while turning while driving. Diagnosed by service as a fault with steering column. This is apparently very common in Chevy Bolts.
4,11742485,chevrolet-bolt-ev-2020,06/06/2026,STRUCTURE,0,0,0,0,0,"After putting washer fluid in, my son closed the hood. The hood did not latch. He tried again but it still didn’t latch. He lifted the hood to find the hood striker had come off the hood. There was nothing to hold the hood closed. I’ve included photos of the striker welds. There is no rust or tearing of the welds. It just separated. It was a bad weld."


time: 3.39 ms (started: 2026-09-26 20:50:08 +05:30)


In [9]:
# complaint_components: the same five complaints, one row per component they name. The one
# that names two components counts once under each.
pd.DataFrame(sql("""
    SELECT *
      FROM complaint_components
     WHERE odi_number IN (SELECT odi_number FROM complaints
                           WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
                           ORDER BY odi_number DESC LIMIT 5)
     ORDER BY odi_number DESC"""))

,odi_number,vehicle_id,component
0,11755357,chevrolet-bolt-ev-2020,STEERING
1,11748422,chevrolet-bolt-ev-2020,FUEL/PROPULSION SYSTEM
2,11745256,chevrolet-bolt-ev-2020,FUEL/PROPULSION SYSTEM
3,11745256,chevrolet-bolt-ev-2020,ELECTRICAL SYSTEM
4,11742825,chevrolet-bolt-ev-2020,STEERING
5,11742485,chevrolet-bolt-ev-2020,STRUCTURE


time: 2.96 ms (started: 2026-09-26 20:51:16 +05:30)


In [11]:
# recalls: `component` is NHTSA's full path; `component_head` is its first part, the same
# vocabulary the complaints use, so it is what the scoreboard matches on.
pd.DataFrame(sql("""
    SELECT campaign_number, vehicle_id, report_date, component_head, component,
           consequence
      FROM recalls
     WHERE vehicle_id = 'chevrolet-bolt-ev-2020'
     ORDER BY campaign_number""", limit=5))

,campaign_number,vehicle_id,report_date,component_head,component,consequence
0,20V184000,chevrolet-bolt-ev-2020,26/03/2020,LATCHES/LOCKS/LINKAGES,LATCHES/LOCKS/LINKAGES:DOORS:LATCH,"If the rear door opens while driving, or the door handle fails to open the rear door, there is an increased risk of injury to the rear passengers."
1,20V808000,chevrolet-bolt-ev-2020,22/12/2020,SERVICE BRAKES,"SERVICE BRAKES, HYDRAULIC:FOUNDATION COMPONENTS:DISC:CALIPER","If a brake caliper fractures and brake fluid is lost, the vehicle may experience reduced brake performance, increasing the risk of a crash."
2,20V811000,chevrolet-bolt-ev-2020,23/12/2020,SEAT BELTS,SEAT BELTS,"If a seat belt assembly is not properly attached to the vehicle, the seat belt may not properly restrain an occupant in the event of a crash, increasing the risk of injury."
3,21V650000,chevrolet-bolt-ev-2020,20/08/2021,ELECTRICAL SYSTEM,ELECTRICAL SYSTEM:PROPULSION SYSTEM:TRACTION BATTERY,A battery fire increases the risk of injury.
4,22V930000,chevrolet-bolt-ev-2020,15/12/2022,STRUCTURE,STRUCTURE:BODY:ROOF AND PILLARS,A vehicle fire can increase the risk of injury.


time: 3.08 ms (started: 2026-09-26 20:52:19 +05:30)
